# Emission and spectra

The lines of a solve, their photon emissivity coefficients split into excitation and recombination, line ratios and a synthetic spectrum.

In [ ]:
import numpy as np
import plasma_plots

import opencr

model = opencr.load_model("hydrogen/atomic/basic", n_max=16)
conditions = opencr.Conditions(2.0, 1e19, densities={"H(n=1)": 1e19, "H+": 1e19})
result = model.solve_steady_state(conditions)

## Lines

`emission()` returns every line with its vacuum wavelength (or air, with `air=True`), emissivity and power. The Balmer lines:

In [ ]:
lines = result.emission(air=True)
balmer = lines.where(lines["lower"] == "H(n=2)", drop=True).isel(line=slice(0, 5))
for line in balmer["line"].values:
    row = balmer.sel(line=line)
    print(f"{str(row['label'].values):6s} {float(row['wavelength']):8.2f} nm  {float(row['emissivity']):.3e} photons/m^3/s")

## Excitation and recombination

In a quasi-static solve each line's emissivity is $n_e \sum_d \mathrm{PEC}_d\, n_d$. At 2 eV excitation from the ground state makes most of H-α, even with as many protons as atoms:

In [ ]:
pec = result.photon_emissivity_coefficients().sel(line="A:3-2")
print(pec.to_series())

Recombination takes over below about 1.2 eV, where excitation across 12 eV dies out:

In [ ]:
data = opencr.scan(model, conditions, electron_temperature=np.geomspace(0.5, 20, 20))
halpha = data["pec"].sel(line="A:3-2")
with plasma_plots.figure(1, 1) as fig:
    for driver, name in (("H(n=1)", "excitation"), ("H+", "recombination")):
        halpha.sel(driver=driver).plasma.plot.lineout(ax=fig[0], title="H-alpha PEC")
        fig[0].lines[-1].set_label(name)
    fig[0].set_xscale("log")
    fig[0].set_yscale("log")
    fig[0].set_ylabel("m^3/s")
    fig[0].legend()

## A line ratio

The Balmer decrement H-α/H-β depends on density, which makes it a diagnostic:

In [ ]:
densities = np.geomspace(1e16, 1e21, 15)
ratio = opencr.scan(model, conditions, effective=False, electron_density=densities)
balmer_ratio = ratio["emissivity"].sel(line="A:3-2") / ratio["emissivity"].sel(line="A:4-2")
with plasma_plots.figure(1, 1) as fig:
    balmer_ratio.plasma.plot.lineout(ax=fig[0], title="Balmer decrement at 2 eV")
    fig[0].set_xscale("log")
    fig[0].set_ylabel("H-alpha / H-beta")

## A synthetic spectrum

The lines Doppler broadened at the gas temperature and an instrumental width of 0.2 nm:

In [ ]:
spectrum = result.emission_spectrum(np.linspace(360.0, 670.0, 4000), fwhm=0.2, air=True)
with plasma_plots.figure(1, 1) as fig:
    spectrum.plasma.plot.lineout(ax=fig[0], title="Balmer series")
    fig[0].set_yscale("log")
    fig[0].set_ylim(spectrum.max() * 1e-5, spectrum.max() * 2)